In [1]:
%pip install playwright google-genai python-dotenv
!python -m playwright install chromium

Note: you may need to restart the kernel to use updated packages.


In [2]:
import asyncio
from dotenv import load_dotenv
from playwright.async_api import async_playwright
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client()

pw = await async_playwright().start()
browser = await pw.chromium.launch(headless=False, slow_mo=700)
context = await browser.new_context(
    user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
)
page = await context.new_page()
print("Chrome is ready!")

Chrome is ready!


In [3]:
from google.genai.errors import ServerError

async def open_amazon() -> str:
    """Opens Amazon India (amazon.in) in the Chrome browser."""
    print("  [CHROME] Opening amazon.in...")
    await page.goto("https://www.amazon.in", wait_until="domcontentloaded")
    return "Amazon India homepage is open."

async def search_product(keyword: str) -> str:
    """Searches for a product keyword on Amazon India."""
    print(f"  [CHROME] Searching Amazon.in for: '{keyword}'")
    await page.wait_for_selector("#twotabsearchtextbox", timeout=10000)
    await page.fill("#twotabsearchtextbox", keyword)
    await page.click("#nav-search-submit-button")
    await page.wait_for_selector("[data-component-type='s-search-result']", timeout=10000)
    return f"Search results loaded for '{keyword}'."

async def get_top_products(limit: int = 5) -> str:
    """Extracts the titles and prices of the top search results on Amazon India."""
    print(f"  [CHROME] Reading top {limit} products...")
    cards = await page.locator("[data-component-type='s-search-result']").all()
    results = []

    for card in cards[:limit]:
        title_loc = card.locator("h2 span").first
        price_loc = card.locator(".a-price .a-offscreen").first
        
        title = (await title_loc.inner_text()).strip() if await title_loc.count() > 0 else "No title"
        price = (await price_loc.inner_text()).strip() if await price_loc.count() > 0 else "Price unavailable"
        results.append(f"- {title} | {price}")

    return "\n".join(results)

tools = [open_amazon, search_product, get_top_products]

goal = """
Open Amazon India, search for 'wireless mouse under 1000', 
check the top 5 products, and recommend the best value option from the list.
"""

# Fallback list so a 503 spike never stalls your live class
models_to_try = ["gemini-3.8-flash", "gemini-3.5-flash", "gemini-3.5-flash-lite"]

for model_name in models_to_try:
    try:
        print(f"Trying {model_name}...")
        chat = client.aio.chats.create(
            model=model_name,
            config=types.GenerateContentConfig(tools=tools, temperature=0.0)
        )
        response = await chat.send_message(goal)
        print(f"\nAgent Recommendation (via {model_name}):\n", response.text)
        break
    except ServerError as e:
        print(f"  [!] {model_name} busy ({e.code}), switching to fallback...")

Trying gemini-3.8-flash...
  [!] gemini-3.8-flash busy (503), switching to fallback...
Trying gemini-3.5-flash...
  [!] gemini-3.5-flash busy (503), switching to fallback...
Trying gemini-3.5-flash-lite...
  [CHROME] Opening amazon.in...
  [CHROME] Searching Amazon.in for: 'wireless mouse under 1000'
  [CHROME] Reading top 5 products...

Agent Recommendation (via gemini-3.5-flash-lite):
 Here are the top 5 wireless mouse options found on Amazon India under ₹1000:

1. **Portronics Toad One Bluetooth Mouse**
   - **Price:** ₹533
   - **Features:** Dual Wireless (2.4 GHz & BT 5.3), 6 Buttons, Rechargeable, RGB Lights, Connects up to 3 Devices, Ergonomic Design.

2. **EvoFox Shadow X Pro Wireless Gaming Mouse**
   - **Price:** ₹999
   - **Features:** Tri-Mode Rechargeable (BT, USB 2.4G & Wired), S203 Sensor, 1000Hz Polling Rate, 6400 DPI, RGB Lighting, 6 Programmable Buttons with Software Support.

3. **Portronics Vader Max Wireless Rechargeable Mouse**
   - **Price:** ₹865
   - **Features